# Python Retest: Model Answers

**Duration: 3 hours (180 minutes) · 20 questions · 200 marks · 10 marks each**

This is a fresh retest modeled on the concepts and difficulty of the original Python Test 1 (core Python + pandas) and Test 2 (database-backed ETL). The business examples and datasets are new; the underlying skills are intentionally similar.

## Suggested timing

| Section | Questions | Suggested time |
|---|---:|---:|
| Core Python | Q1–Q9 | 70 minutes |
| Pandas and files | Q10–Q14 | 45 minutes |
| Database and ETL | Q15–Q20 | 65 minutes |
| **Total** | **20** | **180 minutes** |

Each question is worth 10 marks. Use readable functions, handle edge cases, and follow the explicit requirements. Do not load the full 100,000-row event file into memory. The ETL questions depend on earlier cells, just as the original Test 2 did.

## Data

All fixtures are in `seed_data/data/`. The setup cell below locates the folder from either the project root or the `notebooks/` folder. Read each question fully before coding.

In [ ]:
from pathlib import Path

# Works whether Jupyter is launched from the project root or notebooks/.
CANDIDATE_DIRS = [Path("seed_data/data"), Path("../seed_data/data")]
DATA_DIR = next(path.resolve() for path in CANDIDATE_DIRS if path.exists())
print("Using data directory:", DATA_DIR)

## Shared grading scale

- **10 – Excellent:** correct, complete, follows all constraints, handles required edge cases.
- **7 – Good:** main solution correct with one limited omission.
- **5 – Partial:** meaningful attempt but at least one important requirement is incomplete.
- **2 – Poor:** limited understanding or major implementation error.
- **0 – Blank:** no meaningful attempt.

Use one holistic score per question, not separate marks for every bullet.

Solutions are intentionally straightforward and use familiar Python/pandas/SQLAlchemy patterns. The database cells require the included MySQL Docker environment.

## Solution 1: Question 1: Deduplicate Client Records by Normalized Email

**Concept:** Order-preserving deduplication

**Question summary:** Given:

```python
event_labels = ["LOGIN", "EXPORT", "LOGIN", "UPLOAD", "EXPORT", "DELETE"]
```

Write `dedup_preserve_order(values)` that returns a new list containing only the first occurrence of each value, in the original order. Do not modify the input list. Add a short comment explaining why `list(set(values))` does not meet the order requirement.

In [ ]:
def dedup_clients(records):
    seen = set()
    result = []
    for record in records:
        key = record["email"].strip().casefold()
        if key not in seen:
            seen.add(key)
            result.append(record.copy())
    return result


clients = [
    {"client_id": 1, "email": "Alice@Example.com", "source": "web"},
    {"client_id": 2, "email": "bob@example.com", "source": "app"},
    {"client_id": 3, "email": " alice@example.com ", "source": "import"},
    {"client_id": 4, "email": "cara@example.com", "source": "web"},
]
print(dedup_clients(clients))
# Keep the first full record; a set alone cannot return ordered dictionaries.

## Solution 2: Question 2: Abstract Delivery Channel Interface

**Concept:** Abstract base classes and enforced interfaces

**Question summary:** Using `abc`, define an abstract class `DeliveryChannel` with abstract method `channel_name(self)`. Create `EmailChannel` returning `"email"` and `QueueChannel` returning `"queue"`. Demonstrate that `DeliveryChannel()` raises `TypeError`.

In [ ]:
from abc import ABC, abstractmethod


class DeliveryChannel(ABC):
    @abstractmethod
    def channel_name(self):
        pass


class EmailChannel(DeliveryChannel):
    def channel_name(self):
        return "email"


class QueueChannel(DeliveryChannel):
    def channel_name(self):
        return "queue"


print(EmailChannel().channel_name())
print(QueueChannel().channel_name())
try:
    DeliveryChannel()
except TypeError as error:
    print(error)

## Solution 3: Question 3: Functional Transformation with map()

**Concept:** map() and lambda

**Question summary:** Using `map()` and a `lambda`, create a list of squares for integers from 3 through 18 inclusive. Then write one sentence explaining when a list comprehension would be clearer.

In [ ]:
squares = list(map(lambda number: number ** 2, range(3, 19)))
print(squares)
# A list comprehension is clearer when the expression includes conditions or multiple steps.

## Solution 4: Question 4: Streaming API Event Metrics

**Concept:** Generators and memory-efficient aggregation

**Question summary:** `seed_data/data/api_events.csv` contains 100,000 API event rows with columns `event_id, endpoint, status_code, duration_ms, event_time`. Write `read_api_events(path)` as a generator yielding one dictionary per row. Consume it once and calculate, for each endpoint: total valid events, count of 5xx responses, and average duration in milliseconds. Some rows contain non-numeric status or duration values: skip rows that cannot be safely used and count rejected rows. Do not call `list(read_api_events(...))` and do not store all raw rows.

In [ ]:
import csv
from collections import defaultdict


def read_api_events(path):
    with open(path, "r", newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            yield row


def summarize_api_events(path):
    stats = defaultdict(lambda: {"events": 0, "errors_5xx": 0, "total_ms": 0.0, "duration_count": 0})
    rejected = 0

    for row in read_api_events(path):
        try:
            status = int(row["status_code"])
            duration = float(row["duration_ms"])
        except (KeyError, TypeError, ValueError):
            rejected += 1
            continue

        item = stats[row["endpoint"]]
        item["events"] += 1
        if status >= 500:
            item["errors_5xx"] += 1
        item["total_ms"] += duration
        item["duration_count"] += 1

    result = {}
    for endpoint, item in stats.items():
        result[endpoint] = {
            "events": item["events"],
            "errors_5xx": item["errors_5xx"],
            "average_ms": item["total_ms"] / item["duration_count"],
        }
    return result, rejected


summary, rejected_rows = summarize_api_events(DATA_DIR / "api_events.csv")
print(summary)
print("Rejected rows:", rejected_rows)

## Solution 5: Question 5: Parse Structured API Audit Events with Regex

**Concept:** Regular expressions for structured text

**Question summary:** `seed_data/data/api_audit.log` contains structured lines such as:

```text
2026-11-01T08:03:00Z INFO actor=U104 action=LOGIN object=dashboard message="user action completed"
```

Using regular expressions only, parse each valid line into a dictionary with `timestamp`, `level`, `actor`, `action`, `object`, and `message`. The quoted message can contain spaces. Skip malformed lines and return both the valid records and the skipped-line count.

In [2]:
import re

AUDIT_PATTERN = re.compile(
    r'^(?P<timestamp>\S+)\s+'
    r'(?P<level>DEBUG|INFO|WARNING|ERROR)\s+'
    r'actor=(?P<actor>[A-Za-z0-9_-]+)\s+'
    r'action=(?P<action>[A-Z_]+)\s+'
    r'object=(?P<object>[A-Za-z0-9_:.-]+)\s+'
    r'message="(?P<message>[^"]*)"$',
    re.IGNORECASE,
)


def parse_audit_log(path):
    records = []
    skipped = 0
    with open(path, encoding="utf-8") as file:
        for line in file:
            match = AUDIT_PATTERN.match(line.strip())
            if match:
                records.append(match.groupdict())
            else:
                skipped += 1
    return records, skipped


records, skipped = parse_audit_log(r"C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\api_audit.log")
print(records[:3])
print("Skipped malformed lines:", skipped)

[{'timestamp': '2026-11-01T08:03:00Z', 'level': 'ERROR', 'actor': 'U137', 'action': 'DOWNLOAD', 'object': 'report:monthly', 'message': 'permission check passed'}, {'timestamp': '2026-11-01T08:06:00Z', 'level': 'DEBUG', 'actor': 'U111', 'action': 'LOGIN', 'object': 'report:monthly', 'message': 'user action completed'}, {'timestamp': '2026-11-01T08:09:00Z', 'level': 'DEBUG', 'actor': 'U126', 'action': 'EXPORT', 'object': 'asset:LT-0047', 'message': 'export file generated with warnings'}]
Skipped malformed lines: 3


## Solution 6: Question 6: Repeating-Key Route-Code Encoding

**Concept:** Custom transformation functions and alphabet wraparound

**Question summary:** Implement `encode_route_code(value)`. The input contains lowercase letters `a-z`. Repeat the shift key `[2, 4, 1]` from left to right: the first character moves 2 positions, the second 4, the third 1, then the key repeats. Wrap around from `z` to `a`. Verify that `encode_route_code("xyz")` returns `"zca"`.

In [ ]:
def encode_route_code(value):
    shifts = [2, 4, 1]
    result = []
    for index, char in enumerate(value):
        new_position = (ord(char) - ord("a") + shifts[index % len(shifts)]) % 26
        result.append(chr(ord("a") + new_position))
    return "".join(result)


print(encode_route_code("xyz"))
assert encode_route_code("xyz") == "zca"

## Solution 7: Question 7: Normalize Asset Tags and Clean Reference Numbers

**Concept:** Field normalization and type coercion

**Question summary:** Implement both functions:

1. `normalize_asset_tag(value)` converts variants such as `"lt-0047"`, `" LT 0047 "`, `"LT/0047"` and `"pr.1020"` to canonical `"LT-0047"` or `"PR-1020"`. Return `None` for invalid formats. The format is exactly two letters followed by four digits after ignoring spaces, dots, slashes and hyphens.
2. `clean_reference_code(value)` removes every non-digit character and returns the remaining value as an integer, or `None` if there are no digits.

Demonstrate both functions using at least five inputs each.

In [3]:
import re


def normalize_asset_tag(value):
    match = re.fullmatch(r"\s*([A-Za-z]{2})[\s./-]*(\d{4})\s*", str(value))
    if not match:
        return None
    return match.group(1).upper() + "-" + match.group(2)


def clean_reference_code(value):
    if value is None:
        return None
    digits = re.sub(r"\D", "", str(value))
    return int(digits) if digits else None


asset_tags = ["lt-0047", " LT 0047 ", "LT/0047", "pr.1020", "wrong-tag"]
reference_codes = ["WR-00981", " 45020 ", "12/34/56", "#8823", "n/a"]
print([normalize_asset_tag(value) for value in asset_tags])
print([clean_reference_code(value) for value in reference_codes])

['LT-0047', 'LT-0047', 'LT-0047', 'PR-1020', None]
[981, 45020, 123456, 8823, None]


## Solution 8: Question 8: Refactor File Reading with Logging

**Concept:** Specific exception handling and logging

**Question summary:** Refactor the following poor pattern into `read_text_files(paths)`:

```python
try:
    data = open(path).read()
except:
    print("failed")
```

Use a context manager and UTF-8. Catch `FileNotFoundError` and `PermissionError` separately. Log each attempted path at DEBUG, successful reads at INFO, and failures at WARNING/ERROR. Never use a bare `except` or silently hide failures. Demonstrate one existing file and one missing path.

In [ ]:
import logging

logger = logging.getLogger("retest_file_reader")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    console = logging.StreamHandler()
    console.setFormatter(formatter)
    file_handler = logging.FileHandler("file_reader.log", encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(console)
    logger.addHandler(file_handler)


def read_text_files(paths):
    results = {}
    for path in paths:
        logger.debug("Trying file: %s", path)
        try:
            with open(path, encoding="utf-8") as file:
                results[str(path)] = file.read()
            logger.info("Read successfully: %s", path)
        except FileNotFoundError:
            logger.warning("File not found: %s", path, exc_info=True)
        except PermissionError:
            logger.error("Permission denied: %s", path, exc_info=True)
    return results


files = [DATA_DIR / "api_audit.log", DATA_DIR / "missing_file.txt"]
read_results = read_text_files(files)
print("Successful files:", list(read_results))

## Solution 9: Question 9: TDD and Unit Tests for Quantity Validation

**Concept:** TDD and testable pure functions

**Question summary:** Write `parse_positive_quantity(value)`: accept a positive integer or a string containing a positive whole number; reject booleans, zero, negative numbers, decimal strings, blank strings and non-numeric text by raising `ValueError`. Write at least five pytest-style tests (valid and invalid cases). In 3–4 sentences explain red-green-refactor and why this function is easier to test than code mixed with file or database I/O.

In [ ]:
def parse_positive_quantity(value):
    if isinstance(value, bool):
        raise ValueError("Boolean is not a quantity")
    if isinstance(value, int):
        number = value
    elif isinstance(value, str) and value.strip().isdigit():
        number = int(value.strip())
    else:
        raise ValueError("Enter a whole number")
    if number <= 0:
        raise ValueError("Quantity must be positive")
    return number


# Pytest-style tests. Save these in tests/test_quantity.py to run with pytest.
def test_integer():
    assert parse_positive_quantity(3) == 3


def test_numeric_string():
    assert parse_positive_quantity(" 4 ") == 4


def test_zero_is_invalid():
    import pytest
    with pytest.raises(ValueError):
        parse_positive_quantity(0)


def test_decimal_string_is_invalid():
    import pytest
    with pytest.raises(ValueError):
        parse_positive_quantity("2.5")


def test_boolean_is_invalid():
    import pytest
    with pytest.raises(ValueError):
        parse_positive_quantity(True)


print(parse_positive_quantity("5"))
# Red: write a failing test. Green: make it pass with the smallest change. Refactor: improve code while tests stay green.
# Pure functions are easier to test because the same input gives the same output without files, networks or databases.

## Solution 10: Question 10: Fix Numeric Dtypes in an Operations Survey

**Concept:** Pandas missing-value sentinels and dtype coercion

**Question summary:** Load `seed_data/data/operations_survey.csv` into a DataFrame. The `years_experience` and `annual_pay` columns contain the literal string `"unknown"` in some rows and therefore load as object dtype. Convert both columns to proper numeric dtypes, changing invalid entries to real missing values. Print the resulting dtypes and missing-value counts.

In [6]:
import pandas as pd

survey = pd.read_csv(r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data\operations_survey.csv'
)
for column in ["years_experience", "annual_pay"]:
    survey[column] = pd.to_numeric(survey[column], errors="coerce")

print(survey[["years_experience", "annual_pay"]].dtypes)
print(survey[["years_experience", "annual_pay"]].isna().sum())

years_experience    float64
annual_pay          float64
dtype: object
years_experience    13
annual_pay          10
dtype: int64


## Solution 11: Question 11: Percentage of Remote Workers by Team Size

**Concept:** Pandas grouped aggregation and percentages

**Question summary:** Using the same survey DataFrame, calculate the percentage of respondents whose `remote_work` value is `"Yes"`, grouped by `team_size`. Return one percentage per team-size bucket. Do not use a hard-coded denominator.

In [7]:
remote_percentage = (
    survey["remote_work"].eq("Yes")
    .groupby(survey["team_size"])
    .mean()
    .mul(100)
    .rename("remote_percentage")
    .reset_index()
)
print(remote_percentage.round(2))

         team_size  remote_percentage
0     Large (500+)              54.25
1  Medium (51-500)              52.67
2     Small (1-50)              55.51


## Solution 12: Question 12: Normalize Engagement Type and Report Satisfaction

**Concept:** Exact categorical normalization and two-dimensional reporting

**Question summary:** Normalize `engagement_type_raw` into exactly three values: `CORE`, `FLEX`, or `OTHER` using a function:

- `permanent`, `staff`, `full time`, `ft`, `employee` → `CORE`.
- `contractor`, `agency`, `freelance`, `vendor` → `FLEX`.
- Missing/blank, multi-value, and every unlisted value → `OTHER`.

Strip whitespace and compare lowercase exact values. Add `engagement_group`, show value counts, then report the percentage distribution of `job_satisfaction` for every `(engagement_group, region)` pair.

In [ ]:
CORE_VALUES = {"permanent", "staff", "full time", "ft", "employee"}
FLEX_VALUES = {"contractor", "agency", "freelance", "vendor"}


def normalize_engagement(value):
    if pd.isna(value):
        return "OTHER"
    cleaned = str(value).strip().lower()
    if cleaned in CORE_VALUES:
        return "CORE"
    if cleaned in FLEX_VALUES:
        return "FLEX"
    return "OTHER"


survey["engagement_group"] = survey["engagement_type_raw"].apply(normalize_engagement)
print(survey["engagement_group"].value_counts(dropna=False))
report = pd.crosstab(
    [survey["engagement_group"], survey["region"]],
    survey["job_satisfaction"],
    normalize="index",
).mul(100)
print(report.round(2))

## Solution 13: Question 13: Combine All Sheets from a Workbook

**Concept:** Multi-sheet Excel processing and schema union

**Question summary:** Read every sheet in `seed_data/data/branch_activity.xlsx`. The workbook has one header row per sheet and the sheets do not share identical columns. Add `source_sheet` to each row, combine all sheets into one DataFrame, and write `seed_data/data/branch_activity_combined.csv`. Do not hardcode sheet names or the number of sheets.

In [2]:
DATA_DIR = r'C:\sasivan\Work\Python\Test\Retest\python_retest\seed_data\data'
import pandas as pd
sheets = pd.read_excel(DATA_DIR / "branch_activity.xlsx", sheet_name=None)
frames = []
for sheet_name, frame in sheets.items():
    frame = frame.copy()
    frame["source_sheet"] = sheet_name
    frames.append(frame)

combined = pd.concat(frames, ignore_index=True, sort=False)
output_path = DATA_DIR / "branch_activity_combined.csv"
combined.to_csv(output_path, index=False)
print(combined.head())
print("Saved:", output_path)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

## Solution 14: Question 14: Merge Service Requests and Calculate Client Charges

**Concept:** DataFrame merge, derived columns and aggregation

**Question summary:** Load `service_requests.csv` (`request_id, client_id, service_code, hours`) and `service_catalog.csv` (`service_code, service_name, hourly_rate`). Merge on `service_code`, calculate `line_amount = hours * hourly_rate`, then calculate total service charges and total hours per client, sorted by total charges descending. Do not mutate the original DataFrames.

In [ ]:
requests = pd.read_csv(DATA_DIR / "service_requests.csv")
catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")

merged = requests.merge(
    catalog,
    on="service_code",
    how="left",
    validate="many_to_one",
)
merged["line_amount"] = merged["hours"] * merged["hourly_rate"]
client_report = (
    merged.groupby("client_id", as_index=False)
    .agg(
        total_charges=("line_amount", "sum"),
        total_hours=("hours", "sum"),
    )
    .sort_values("total_charges", ascending=False)
)
print(client_report.head())

## Solution 15: Question 15: Config-Driven MySQL Connection and Persistent Logging

**Concept:** Configuration, SQLAlchemy and logging

**Question summary:** Connect to the `python_retest` MySQL database using `pymysql` via SQLAlchemy. Store host, port, username, password and database in a configuration dictionary or environment variables; do not inline them inside `create_engine`. Configure one logger that writes to both console and `etl_run.log`. Catch and log a connection failure, then re-raise it. Query and display all rows from the seeded `region_lookup` table.

In [ ]:
import logging
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

DB_CONFIG = {
    "host": os.getenv("PY_RETEST_DB_HOST", "127.0.0.1"),
    "port": int(os.getenv("PY_RETEST_DB_PORT", "3307")),
    "user": os.getenv("PY_RETEST_DB_USER", "root"),
    "password": os.getenv("PY_RETEST_DB_PASSWORD", "PythonRetest_2026!"),
    "database": os.getenv("PY_RETEST_DB_NAME", "python_retest"),
}

logger = logging.getLogger("python_retest_etl")
logger.setLevel(logging.INFO)
if not logger.handlers:
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    console = logging.StreamHandler()
    console.setFormatter(formatter)
    file_handler = logging.FileHandler("etl_run.log", encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(console)
    logger.addHandler(file_handler)

url = URL.create(
    "mysql+pymysql",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"],
    database=DB_CONFIG["database"],
)
try:
    engine = create_engine(url, pool_pre_ping=True)
    with engine.connect() as connection:
        region_lookup = pd.read_sql(text("SELECT country_code, region_name FROM region_lookup"), connection)
    logger.info("Connected and loaded %d region lookup rows", len(region_lookup))
    print(region_lookup)
except Exception:
    logger.exception("Could not connect to the retest database")
    raise

## Solution 16: Question 16: Extract Two Raw Sources Without Cleaning

**Concept:** ETL extraction and source preservation

**Question summary:** Read `seed_data/data/raw_clients.csv` and `seed_data/data/raw_jobs.json` into two separate Python structures (list of dictionaries or DataFrames). This stage is extraction only: preserve the incomplete customer row and the non-numeric job amount; do not clean or drop anything. Catch missing-file and malformed-JSON errors, log before re-raising, and log the number of rows extracted from each source.

In [ ]:
import csv
import json


def extract_raw_files(data_dir=DATA_DIR):
    try:
        with open(data_dir / "raw_clients.csv", newline="", encoding="utf-8") as file:
            raw_clients = list(csv.DictReader(file))
        with open(data_dir / "raw_jobs.json", encoding="utf-8") as file:
            raw_jobs = json.load(file)
    except FileNotFoundError:
        logger.exception("An ETL source file is missing")
        raise
    except json.JSONDecodeError:
        logger.exception("The jobs JSON is malformed")
        raise

    logger.info("Extracted %d clients and %d jobs", len(raw_clients), len(raw_jobs))
    return raw_clients, raw_jobs


raw_clients, raw_jobs = extract_raw_files()

## Solution 17: Question 17: Clean the Extracted Client and Job Data

**Concept:** ETL transformation, numeric coercion and rejection counts

**Question summary:** Using the structures from Q16:

- Drop client records whose `client_id` is missing or blank, and log a WARNING with the rejected count.
- Convert job `amount` to numeric; drop rows that cannot be converted and log the affected job ID(s) or count.
- Keep valid rows and report retained and rejected counts for both datasets.

Do not silently modify the raw structures from Q16.

In [ ]:
def clean_extracted_data(raw_clients, raw_jobs):
    clients = pd.DataFrame(raw_clients).copy()
    jobs = pd.DataFrame(raw_jobs).copy()

    client_before = len(clients)
    clients["client_id"] = clients["client_id"].astype("string").str.strip()
    clients = clients[clients["client_id"].notna() & clients["client_id"].ne("")].copy()

    job_before = len(jobs)
    jobs["amount"] = pd.to_numeric(jobs["amount"], errors="coerce")
    bad_jobs = jobs[jobs["amount"].isna()]
    jobs = jobs.dropna(subset=["amount"]).copy()

    logger.warning("Dropped %d clients with missing IDs", client_before - len(clients))
    logger.warning("Dropped %d jobs with invalid amounts: %s", job_before - len(jobs), bad_jobs.get("job_id", pd.Series(dtype=int)).tolist())
    logger.info("Clients kept=%d/%d; jobs kept=%d/%d", len(clients), client_before, len(jobs), job_before)
    return clients, jobs


clean_clients, clean_jobs = clean_extracted_data(raw_clients, raw_jobs)

## Solution 18: Question 18: Enrich Clients from the MySQL Region Lookup

**Concept:** ETL enrichment and reference-data joins

**Question summary:** Use the `region_lookup` table queried in Q15 (or query it again). Join it to the cleaned clients on `country_code`, adding `region_name`. Every valid country code in this fixture has a match. Use a left merge or dictionary lookup, verify that no `region_name` is missing, and log the enriched row count. If a required field or lookup is missing, log the error and re-raise it.

In [ ]:
def enrich_clients(clean_clients, engine):
    try:
        with engine.connect() as connection:
            lookup = pd.read_sql(text("SELECT country_code, region_name FROM region_lookup"), connection)
        enriched = clean_clients.merge(
            lookup,
            on="country_code",
            how="left",
            validate="many_to_one",
        )
        if enriched["region_name"].isna().any():
            missing = enriched.loc[enriched["region_name"].isna(), "country_code"].unique().tolist()
            raise ValueError(f"Missing region lookup for country codes: {missing}")
        logger.info("Enriched %d client records", len(enriched))
        return enriched
    except Exception:
        logger.exception("Client enrichment failed")
        raise


enriched_clients = enrich_clients(clean_clients, engine)
print(enriched_clients.head())

## Solution 19: Question 19: Idempotent Load and Round-Trip Validation

**Concept:** Safe database loading and read-back verification

**Question summary:** Create `clean_clients` in `python_retest` if it does not exist, with `client_id` as the primary key and columns for `name`, `email`, `country_code`, and `region_name`. Load the enriched clients using `INSERT ... ON DUPLICATE KEY UPDATE` keyed on `client_id`, so repeating the same batch does not duplicate rows. Read the table back into a DataFrame. Assert that the row count matches the batch and verify at least one known client's email and region. Log success; log and re-raise database errors.

In [ ]:
from sqlalchemy import text


def load_and_validate_clients(enriched_clients, engine):
    create_sql = text("""
        CREATE TABLE IF NOT EXISTS clean_clients (
            client_id VARCHAR(10) PRIMARY KEY,
            name VARCHAR(100) NOT NULL,
            email VARCHAR(150),
            country_code CHAR(2) NOT NULL,
            region_name VARCHAR(50) NOT NULL
        )
    """)
    upsert_sql = text("""
        INSERT INTO clean_clients (client_id, name, email, country_code, region_name)
        VALUES (:client_id, :name, :email, :country_code, :region_name)
        ON DUPLICATE KEY UPDATE
            name = VALUES(name),
            email = VALUES(email),
            country_code = VALUES(country_code),
            region_name = VALUES(region_name)
    """)
    records = enriched_clients[["client_id", "name", "email", "country_code", "region_name"]].to_dict("records")

    try:
        with engine.begin() as connection:
            connection.execute(create_sql)
            connection.execute(upsert_sql, records)
        with engine.connect() as connection:
            loaded = pd.read_sql(text("SELECT * FROM clean_clients"), connection)
        assert len(loaded) == len(records)
        expected = records[0]
        actual = loaded.loc[loaded["client_id"] == expected["client_id"]].iloc[0]
        assert actual["email"] == expected["email"]
        assert actual["region_name"] == expected["region_name"]
        logger.info("Round-trip validation passed for %d clients", len(loaded))
        return loaded
    except Exception:
        logger.exception("Loading or round-trip validation failed")
        raise


loaded_clients = load_and_validate_clients(enriched_clients, engine)
print(loaded_clients.head())

## Solution 20: Question 20: Refactor the Work into a Testable ETL Pipeline

**Concept:** ETL orchestration, modularity and independent testing

**Question summary:** Refactor Q16–Q19 into:

- `extract()` — reads both raw files and returns their raw structures.
- `transform(...)` — cleans clients and jobs, queries the region lookup and returns enriched clients plus cleaned jobs.
- `load(...)` — creates/updates `clean_clients` using the same idempotent strategy and validates the result.
- `main()` — calls the stages in order and logs progress before and after each stage.

Call `main()`. Then write 4–6 sentences explaining why independent ETL stages improve testing, debugging and maintenance.

In [ ]:
def extract():
    return extract_raw_files(DATA_DIR)


def transform(raw_clients, raw_jobs, engine):
    clean_clients, clean_jobs = clean_extracted_data(raw_clients, raw_jobs)
    enriched_clients = enrich_clients(clean_clients, engine)
    return enriched_clients, clean_jobs


def load(enriched_clients, engine):
    return load_and_validate_clients(enriched_clients, engine)


def main():
    logger.info("ETL started")
    logger.info("Starting extract")
    raw_clients, raw_jobs = extract()
    logger.info("Extract complete")

    logger.info("Starting transform")
    enriched, clean_jobs = transform(raw_clients, raw_jobs, engine)
    logger.info("Transform complete")

    logger.info("Starting load")
    loaded = load(enriched, engine)
    logger.info("Load and validation complete")
    logger.info("ETL finished successfully")
    return loaded, clean_jobs


# Run only when the MySQL container is running and seed_mysql.py has been executed.
# loaded_clients_final, clean_jobs_final = main()

# Written answer: Extract, transform and load have separate responsibilities and can be tested independently.
# A failure can be isolated to reading, cleaning/enrichment or database persistence rather than debugging one large script.
# Pure transformations can use small in-memory test data and do not need a file system or database mock.
# Each stage can be changed without rewriting unrelated stages, which improves maintenance and reuse.
# The orchestrator makes execution order and progress visible through consistent logging.